# Module 4: Memory and Durable Sessions

An agent that forgets every turn is not useful. In this module, you will create an agent and give it two kinds of memory:

1. **Session memory** so the agent remembers the current conversation:
   - **In-process memory** for remembering within a single run
   - **File-backed session memory** so conversation history survives a restart
   - **Akamai Object Storage** so any agent replica behind a NodeBalancer can pick up the same conversation

2. **Long-term user memory** so the agent remembers preferences and facts across conversations, not just within the current one.

As in previous sessions, you will build it by hand first, then with [Strands](https://strandsagents.com).

## Learning objectives
- Learn why a fresh agent has no memory of previous turns
- Keep conversation memory in process by hand, then by reusing one Strands agent
- Understand why in-process memory is lost on restart or across replicas
- Persist and rehydrate a session by hand, then with a Strands `SessionManager`
- Store sessions on Akamai Object Storage so they survive restarts and scale across pods
- Give the agent long-term memory of the user with tools and a SQLite store

## Prerequisites
- Finished Modules 1 to 3
- The same model endpoint from Module 1
- For the Object Storage section (optional): a Linode token and the will to create a bucket
- About 40 minutes

References: [Strands sessions and state](https://strandsagents.com/docs/user-guide/concepts/agents/state/) · [Akamai Object Storage](https://www.linode.com/products/object-storage/) · [boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)

## Memory design basics

By default a language model is stateless. Each request is judged only on the messages you send it. An
agent "remembers" only because something keeps the past messages and sends them again. The
question is *where* that history lives:

- **In process:** a Python object like an array that lives in memory. Fast, but gone when the process exits, and not
  shared with any other process.
- **On a volume:** a file on disk. Survives a restart, but only on that one machine.
- **On Object Storage:** an S3-compatible bucket. Survives restarts and is shared, so any
  replica behind a NodeBalancer can rehydrate the same conversation.

![Three places conversation memory can live: in process, on a file volume, and on Akamai Object Storage shared across replicas](../images/04_sessions_architecture.png)

## 1. Setup

`httpx` covers the by-hand sections, `boto3` is needed only for the Object Storage section,
and Strands provides the session managers.

In [1]:
!pip install -q httpx boto3 python-dotenv "strands-agents[openai]" strands-agents-tools


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Configure the model

Using the same values from Module 1 we will create a `chat()` function to handle the raw request throughout this module.

In [2]:
import os, json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL_ID = os.getenv("VLLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
API_KEY = os.getenv("VLLM_API_KEY", "placeholder")

SYSTEM_PROMPT = "You are an Akamai Cloud Solutions Architect. Be concise."

def chat(messages, temperature=0.2):
    """POST to the vLLM chat endpoint and return the raw JSON (from Module 1)."""
    resp = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": MODEL_ID, "messages": messages, "temperature": temperature},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

print("Model:", MODEL_ID)

Model: Qwen/Qwen2.5-7B-Instruct


## 3. Understanding why models have no memory

In this example we send the model two independent requests. Notice how the second request has no idea about the first one. This is because nothing carried the history forward. The model is stateless y default. 

In [3]:
turn1 = chat([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "My name is Du'An and I run a GPU cluster in us-sea."},
])
print("Turn 1:", turn1["choices"][0]["message"]["content"])

turn2 = chat([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is my name, and where is my cluster?"},
])
print("Turn 2:", turn2["choices"][0]["message"]["content"])

Turn 1: Hello Du'An! How can I assist you with your GPU cluster in us-sea? Do you need help with optimization, scaling, security, or any specific issues?
Turn 2: Your name is not provided in the request. Your cluster location would depend on where it was deployed. Can you provide more details about your cluster (e.g., cloud provider, region)?


## 4. In-Process Memory

For the model to remember a conversation, our code needs to keep past messages and send them again on each turn.

Below, we use a `history` list that stores alternating user and assistant messages. Each time the user sends a message, the agent appends the new user message, the assistant response, any `tool_calls`, and any `tool_results`. On the next turn, that full history is sent back to the model so it has the conversation context and a record of the actions it has taken.

In [4]:
def remember(history, question, temperature=0.2):
    # Add the new user turn, send the WHOLE history, then store the answer
    history.append({"role": "user", "content": question})
    answer = chat(history, temperature=temperature)["choices"][0]["message"]["content"]
    history.append({"role": "assistant", "content": answer})
    return answer

history = [{"role": "system", "content": SYSTEM_PROMPT}]
print("Turn 1:", remember(history, "My name is Du'An and I run a GPU cluster in us-sea."))
print("Turn 2:", remember(history, "What is my name, and where is my cluster?"))
print(f"\nThe history now holds {len(history)} messages.")

Turn 1: Hello Du'An! How can I assist you with your GPU cluster in us-sea? Do you need help with optimization, scaling, security, or any specific issues?
Turn 2: Your name is Du'An, and your cluster is in us-sea (likely referring to the US East region). How can I assist you further with your cluster?

The history now holds 5 messages.


Most agentic frameworks handle conversation history for you. When we reuse a single Strands agent across turns, it stores each user and assistant message in `.messages`, which accumulates automatically.

Because we are using vLLM for inference, the agent needs at least one tool, as discussed in Module 1. For this example, we give it the `current_time` tool.

In [7]:
from strands import Agent
from strands.models.openai import OpenAIModel
from strands_tools import current_time

model = OpenAIModel(
    client_args={"api_key": API_KEY, "base_url": BASE_URL},
    model_id=MODEL_ID,
    params={"temperature": 0.2},
)

# Reuse ONE agent object across turns.
agent = Agent(model=model, tools=[current_time], system_prompt=SYSTEM_PROMPT)
print("Turn 1:")
_ = agent("My name is Du'An and I run a GPU cluster in us-sea.")

print("\nTurn 2:")
_ = agent("What is my name, and where is my cluster?")
print(f"\n\nThe agent is holding {len(agent.messages)} messages in memory.")

Turn 1:
It's great to hear about your GPU cluster! To better assist you, could you please specify what kind of assistance you need regarding your cluster in us-sea? Are you looking for optimization tips, monitoring solutions, or something else?
Turn 2:
Your name is Du'An, and your cluster is located in us-sea (likely referring to the US West Coast, possibly in a specific data center or cloud region like AWS us-west-1 or another provider's equivalent).

The agent is holding 4 messages in memory.


Notice how `agent(prompt)` does two things: it invokes the model and stores the user and assistant messages from that turn. The agent keeps that conversation history in memory until the Python process ends, you create a new agent instance, or you reset the agent.


## 5. In-Process Memory Does Not Persist

The agent messages or `memory` lives inside the agent object. If you restart the process, create a new agent, or send the next request to a different replica, you get a brand-new agent that starts empty.

Below we create a new agent that displays the new process or replica example.

In [9]:
new_agent = Agent(model=model, tools=[current_time], system_prompt=SYSTEM_PROMPT)
print(str(new_agent("What is my name, and where is my cluster?")))

I don't have access to your personal information or details about your cluster. Can you provide more context or specify what cluster you're referring to?I don't have access to your personal information or details about your cluster. Can you provide more context or specify what cluster you're referring to?



## 6. Durable Sessions

The `history` list from section 4 gives us memory only while the process is running. To survive a runtime restart, that history has to live outside the process.

Below, we save the same `history` list to disk under a `session_id`. A fresh process can load that file, pass the restored history back into `remember()`, and continue the conversation.

In [11]:
def save_session(session_id, history, store="session_store"):
    """Save a session to disk, keyed by session_id."""
    os.makedirs(store, exist_ok=True)
    with open(f"{store}/{session_id}.json", "w") as f:
        json.dump(history, f)

def load_session(session_id, store="session_store"):
    """Load a session from disk, or return a fresh one if it does not exist."""
    path = f"{store}/{session_id}.json"
    if os.path.exists(path):
        return json.load(open(path))
    return [{"role": "system", "content": SYSTEM_PROMPT}]

session_id = "user-123"

# Process 1: load (empty), have a turn, save to disk
history = load_session(session_id)
# Remember this was created in step 4.
remember(history, "I work for Akamai. My favorite programming language is Python.")
save_session(session_id, history)
print(f"saved {len(history)} messages to disk")

# Process 2 (a restart, or a second replica): load from disk, the memory is back
history = load_session(session_id)
print(f"loaded {len(history)} messages from disk")
print("Answer:", remember(history, "Where do I work? What is my favorite language?"))

saved 5 messages to disk
loaded 5 messages from disk
Answer: You work for Akamai, and your favorite programming language is Python. How can I assist you further?


Strands wraps this same pattern in a `SessionManager`. Give an agent a `FileSessionManager` with a `session_id`, and Strands persists and reloads the conversation history for you.

Create a second agent with the same `session_id`, and it picks up where the first one left off. In this example, that second agent stands in for a restarted process or another replica handling the next turn.

In [16]:
from strands.session.file_session_manager import FileSessionManager

# Agent A persists its conversation to disk, keyed by session_id
agent_a = Agent(
    model=model,
    tools=[current_time],
    session_manager=FileSessionManager(session_id="user-456", storage_dir="session_store"),
    system_prompt=SYSTEM_PROMPT,
)
print("Turn 1:")
_ = agent_a("My name is Du'An and I run a GPU cluster in us-sea.")

# Agent B is a brand-new agent (a restart, or a second pod). Same session_id, so it rehydrates.
agent_b = Agent(
    model=model,
    tools=[current_time],
    session_manager=FileSessionManager(session_id="user-456", storage_dir="session_store"),
    system_prompt=SYSTEM_PROMPT,
)

print("\n\nTurn 2:")
_ = agent_b("What is my name, and where is my cluster?")

Turn 1:
Nice to meet you, Du'An! To better assist you, could you please specify what exactly you need help with regarding your GPU cluster in us-sea? Are you looking for performance optimization, security measures, or perhaps monitoring solutions?

Turn 2:
Your name is Du'An, and your cluster is located in `us-sea`.

After running the code above, open `./session_store/session_user-456` to see what Strands persisted. The directory contains the agent’s session state and the message history it rehydrates on the next run.

## 7. Storing Sessions on Akamai Object Storage (Optional)

Storing sessions in files works on one machine. To survive a restart and share sessions across replicas, the session history needs to live in a durable shared store. For this workshop, we use Akamai Object Storage, which is S3-compatible.

Strands ships an `S3SessionManager` for any S3-compatible object store. In this example, we wrap it in a small Akamai-native manager so the configuration uses Akamai terms: a **cluster** such as `us-sea-1`, a bucket, and your Object Storage access keys. The wrapper points the session manager at your bucket without requiring AWS environment variables.

Now the agent’s memory lives in your Akamai cloud account, and any pod behind a NodeBalancer can rehydrate the same `session_id`.

This section is optional because it creates a real bucket. The provisioning script creates the bucket, creates an access key limited to that bucket, and prints the environment variables to add to your `.env`.


In [17]:
# Create the Akamai Object Storage bucket
!python ./scripts/create_bucket.py

Bucket 'akamai-sa-agent-sessions' already exists (akamai-sa-agent-sessions.us-sea-1.linodeobjects.com); reusing it.

Add these to your .env, then re-run section 7 of the notebook:

SESSION_BUCKET=akamai-sa-agent-sessions
SESSION_CLUSTER=us-sea-1
SESSION_ACCESS_KEY=JNZVTMW5HK9V7WMHVGHW
SESSION_SECRET_KEY=R34iUF59G7eHsQPvmFWfUDsnNmyRZ8rRE2nvsqSR

This key is limited to the 'akamai-sa-agent-sessions' bucket (read/write).
This bucket is billed until you delete it (scripts/delete_bucket.py or the Cloud Manager).


>NOTE: After the script finishes, add `SESSION_BUCKET`, `SESSION_CLUSTER`, `SESSION_ACCESS_KEY`, and `SESSION_SECRET_KEY` to your .env, then reload the environment and run the Object Storage demo

In [19]:
# NOTE: This cell runs the demo if `SESSION_BUCKET` is set, and skips cleanly otherwise
# so the rest of the notebook still runs.
SESSION_BUCKET = os.getenv("SESSION_BUCKET")
SESSION_CLUSTER = os.getenv("SESSION_CLUSTER")        # the Object Storage cluster, e.g. us-sea-1
SESSION_ACCESS_KEY = os.getenv("SESSION_ACCESS_KEY")  # your Akamai Object Storage keys, passed explicitly
SESSION_SECRET_KEY = os.getenv("SESSION_SECRET_KEY")

if SESSION_BUCKET:
    import boto3
    from strands.session.s3_session_manager import S3SessionManager

    class AkamaiObjectStorageSessionManager(S3SessionManager):
        """Store Strands sessions on Akamai Object Storage.

        Object Storage is S3-compatible, so this is a thin wrapper over S3SessionManager: give
        it a cluster (like us-sea-1) and your Object Storage keys, and it derives the endpoint
        and passes the credentials. boto3 names the parameters aws_*, but the keys are your
        Akamai keys; there are no AWS_* environment variables.
        """
        def __init__(self, session_id, bucket, cluster, access_key, secret_key, **kwargs):
            akamai = boto3.Session(aws_access_key_id=access_key, aws_secret_access_key=secret_key)
            super().__init__(
                session_id=session_id,
                bucket=bucket,
                endpoint_url=f"https://{cluster}.linodeobjects.com",
                region_name=cluster,
                boto_session=akamai,
                **kwargs,
            )

    def akamai_session(sid):
        return AkamaiObjectStorageSessionManager(
            session_id=sid,
            bucket=SESSION_BUCKET,
            cluster=SESSION_CLUSTER,
            access_key=SESSION_ACCESS_KEY,
            secret_key=SESSION_SECRET_KEY,
        )

    # Agent A writes its memory to Object Storage
    print("Turn 1:")
    agent_a = Agent(model=model, tools=[current_time], session_manager=akamai_session("user-789"), system_prompt=SYSTEM_PROMPT)
    _ = agent_a("I work for Akamai. My favorite programming language is Python.")

    print("\n\nTurn 2:")
    # Agent B, a fresh agent, rehydrates the same session from Object Storage
    agent_b = Agent(model=model, tools=[current_time], session_manager=akamai_session("user-789"), system_prompt=SYSTEM_PROMPT)
    _ = agent_b("Where do I work, and what is my favorite language?")
    print(f"\nSession is stored in Akamai Object Storage bucket: {SESSION_BUCKET}")
else:
    print("SESSION_BUCKET is not set, so the Object Storage demo was skipped.")
    print("To run it for real:")
    print("  1. python workshop/04_memory_and_sessions/scripts/create_bucket.py")
    print("  2. set SESSION_BUCKET, SESSION_CLUSTER, SESSION_ACCESS_KEY, SESSION_SECRET_KEY")
    print("     in your .env, then re-run this cell.")

Turn 1:
Great to know, Du'An! As someone who works for Akamai and favors Python, you might find our Akamai Edge Computing solutions particularly interesting. Would you like more information on how Python can be used with Akamai technologies or any specific projects you're working on?

Turn 2:
You work for Akamai, and your favorite programming language is Python.
Session is stored in Akamai Object Storage bucket: akamai-sa-agent-sessions


### Why This Matters for Production

In production, the agent runs as an HTTP service on Linode Kubernetes Engine, often with multiple pods behind a NodeBalancer. The NodeBalancer might send turn 1 to pod A and turn 2 to pod B.

With in-process memory, pod B has no conversation history. With sessions on Object Storage, every pod rehydrates the same `session_id` from the same bucket, so the conversation continues no matter which pod handles the request.

That is the difference between a demo and a service.

>This pattern assumes one in-flight request per `session_id`. If two pods handle the same session at the same time, they can race: both load the same history, append different messages, and the last writer may overwrite the other. Production systems usually prevent concurrent writes per session with request routing, a per-session lock, optimistic concurrency, or a database/session service that supports conditional updates.

## 8. Long-Term Memory: Remembering the User

Sessions remember the *conversation*. They also grow over time: every turn adds more messages, which means more tokens on each request and more context being sent to the language model.

Strands provides conversation managers to control that growth. A `SlidingWindowConversationManager` drops older messages from the message array, while a `SummarizingConversationManager` compresses older history into a summary.

That helps with long conversations, but it creates a different problem: the agent may lose important user details when old messages are removed, or when the user starts a new session. Close the session, trim the history, or come back tomorrow, and the agent may forget your name, region, and setup.

Long-term memory solves that problem. It is a small, durable set of facts and preferences about the user, kept across conversations.

Two simple memory approaches cover most cases:

- **Preferences** are keyed: one value per key, and a new value replaces the old. A `preferred_region` can move from `us-ord` to `us-sea` cleanly.
- **Facts** accumulate, grouped by category: identity, infrastructure, projects.

In this example, the agent manages long-term memory with tools. It stores a preference or fact when it learns one, and recalls those details when it needs them. No separate extraction pass, no vector database.

### Creating the Memory Store

Long-term memory needs a store the agent can query. A vector database is a common choice, but it is overkill for one user's handful of facts. In this section, we use SQLite from the Python standard library: no new dependency, no server.

In production, this local database would become a durable shared store, such as managed PostgreSQL on Akamai. That gives every pod access to the same user memory, the same upgrade we made for sessions. At larger scale, you can add `pgvector` or a memory service like `mem0` for semantic search across many memories.

In [20]:
import sqlite3

MEMORY_DB = "memory.db"
USER_ID = "duan-123"  # whose memory this is. In production it arrives on the request, like a session id.

def memory_db():
    conn = sqlite3.connect(MEMORY_DB)
    # Preferences are keyed: one value per key, a new value replaces the old.
    conn.execute("CREATE TABLE IF NOT EXISTS preferences (user_id TEXT, key TEXT, value TEXT, PRIMARY KEY (user_id, key))")
    # Facts accumulate, grouped by category.
    conn.execute("CREATE TABLE IF NOT EXISTS facts (user_id TEXT, category TEXT, fact TEXT, UNIQUE (user_id, category, fact))")
    return conn

print("memory store ready:", MEMORY_DB)

memory store ready: memory.db


### Creating the Agent's Memory Tools

Now we will give the agent a few tools for working with long-term memory. The agent can call `set_preference` and `remember_fact` when it learns something worth keeping, and `recall` when it needs to look something up.

The docstring on each tool is key. It is the instruction the model reads when deciding whether to call the tool, so write it like you are explaining the tool to the agent.

In [21]:
from strands import tool

@tool
def set_preference(key: str, value: str) -> str:
    """Save or update a durable user preference, for example preferred_region or comms_style.

    Use this when the user states a lasting preference. One value per key, so a new value
    replaces the old one.
    Args:
        key: The key to save the preference under.
        value: The value to save for the preference.
    Returns:
        A message indicating that the preference was saved.
    """
    with memory_db() as c:
        c.execute(
            "INSERT INTO preferences (user_id, key, value) VALUES (?, ?, ?) "
            "ON CONFLICT(user_id, key) DO UPDATE SET value = excluded.value",
            (USER_ID, key, value),
        )
    return f"Saved preference: {key} = {value}"

@tool
def remember_fact(fact: str, category: str = "general") -> str:
    """Save a durable fact about the user, tagged with a category like identity, infrastructure, or project.

    Use this when you learn something worth keeping across conversations.
    Args:
        fact: The fact to save.
        category: The category to save the fact under.
    Returns:
        A message indicating that the fact was saved.
    """
    with memory_db() as c:
        c.execute("INSERT OR IGNORE INTO facts (user_id, category, fact) VALUES (?, ?, ?)", (USER_ID, category, fact))
    return f"Remembered [{category}]: {fact}"

@tool
def recall(query: str = "") -> str:
    """Recall what you know about the user. Pass a keyword to filter the facts, or nothing to get everything.
    Args:
        query: A keyword to filter the facts, or an empty string to get everything.
    Returns:
        A string listing the preferences and facts.
    """
    with memory_db() as c:
        prefs = c.execute("SELECT key, value FROM preferences WHERE user_id = ?", (USER_ID,)).fetchall()
        if query:
            facts = c.execute("SELECT category, fact FROM facts WHERE user_id = ? AND fact LIKE ?", (USER_ID, f"%{query}%")).fetchall()
        else:
            facts = c.execute("SELECT category, fact FROM facts WHERE user_id = ?", (USER_ID,)).fetchall()
    lines = [f"{key}: {value}" for key, value in prefs] + [f"[{cat}] {fact}" for cat, fact in facts]
    return "\n".join(lines) if lines else "Nothing stored about this user yet."

### Store: Verify the Agent Stores Memory on Its Own

Below, we give the agent a system prompt that tells it to save durable preferences and facts as it learns them. When you share something worth remembering, watch the agent call `set_preference` or `remember_fact` without being asked.

Then inspect the store to see what it saved.

In [23]:
SYSTEM_PROMPT = (
    "You are an Akamai Cloud Solutions Architect. "
    "When the user states a durable preference, call set_preference. "
    "When you learn a durable fact about them, call remember_fact. "
    "Save memory proactively as you learn it, without being asked."
)

writer = Agent(model=model, tools=[set_preference, remember_fact, recall], system_prompt=SYSTEM_PROMPT)
_ = writer("I prefer the us-sea region, and I run a vLLM GPU cluster for inference.")

# Look at what the agent chose to store.
with memory_db() as c:
    print("\n\nRecalling the user's memory:")
    print("\npreferences:", c.execute("SELECT key, value FROM preferences").fetchall())
    print("facts:      ", c.execute("SELECT category, fact FROM facts").fetchall())


Tool #1: set_preference

Tool #2: remember_fact
Your preference for the us-sea region has been saved, and I've noted that you run a vLLM GPU cluster for inference. Is there anything else specific you'd like to add or discuss regarding your infrastructure setup?

Recalling the user's memory:

preferences: [('preferred_region', 'us-sea')]
facts:       [('infrastructure', 'run a vLLM GPU cluster for inference')]


### Recall: Bring Memory Back into Context

There are two ways to bring long-term memory back into the conversation, and you usually want both.

The `recall` tool is for explicit lookups. If the user asks what the agent knows, the agent can query the memory store and answer from it. This works, but a small model will not always remember to call the tool at the right time.

The more reliable pattern is to inject known user memory into the system prompt before the turn. That way, the agent starts with the important facts already in context. You load that memory when you build the agent, the same place you set the `session_id`.

In [25]:
# 1) The recall tool: a fresh agent looks up what it knows.
print(">>> recall tool")
reader = Agent(
    model=model,
    tools=[recall],
    system_prompt="You are an Akamai Cloud Solutions Architect. Use the recall tool to look up what you know about the user.",
)
_ = reader("What do you remember about me?")

# 2) The reliable pattern: inject known memory into the system prompt at build time, so the
#    agent always knows the user instead of depending on it to call recall.
def known_memory(user_id):
    """Everything stored about a user, formatted for the system prompt."""
    with memory_db() as c:
        prefs = c.execute("SELECT key, value FROM preferences WHERE user_id = ?", (user_id,)).fetchall()
        facts = c.execute("SELECT fact FROM facts WHERE user_id = ?", (user_id,)).fetchall()
    return "; ".join([f"{key}={value}" for key, value in prefs] + [fact for (fact,) in facts])

def memory_agent(user_id):
    return Agent(
        model=model,
        tools=[set_preference, remember_fact, recall],
        system_prompt=SYSTEM_PROMPT + f"\n\nWhat you already know about this user: {known_memory(user_id)}",
    )

print("\n\n>>> memory injected into the system prompt")
agent = memory_agent(USER_ID)  # a brand-new agent that already knows the user
_ = agent("I am planning a new inference deployment. Where should I put it, and why?")

>>> recall tool

Tool #1: recall
I remember that your preferred region is US-SEA. Additionally, you mentioned running a vLLM GPU cluster for inference. Is there anything specific you need help with regarding this setup?

>>> memory injected into the system prompt
Given your preference `preferred_region=us-sea`, I recommend deploying your vLLM GPU cluster in the US-West (California) region. This region is optimal due to its proximity to many users on the West Coast of North America, which can reduce latency and improve performance for your inference tasks.

Let's set this as your preferred region to ensure future deployments use this setting.


Tool #1: set_preference
Your preferred region has been set to `us-west`. For your new inference deployment, using the US-West region will help minimize latency and enhance performance for your users on the West Coast and nearby areas. If you need further assistance with setting up the vLLM GPU cluster, feel free to ask!

> **Note:** `recall` is useful, but it depends on the model choosing to call the tool. For important user context, do not rely on that.
>
> The reliable pattern is to load known memory before the turn and inject it into the system prompt. Then the agent starts with the user’s preferences and facts already in context.
>
> Use both patterns together: tools let the agent store and look up memory, and prompt injection makes the most important memory available every turn.

## Try It Yourself

1. **Prove sessions survive a restart.** Have an agent learn a fact with a `FileSessionManager`. Restart the kernel, rerun the setup cells, then create a fresh agent with the same `session_id`. Confirm it still knows the fact.
2. **Run two conversations at once.** Create two agents with different `session_id` values and confirm each one remembers only its own facts.
3. **Inspect the session store.** Open `session_store/` and look at what Strands wrote. The messages are stored on disk.
4. **Update a preference.** Tell the agent your region changed to `us-ord`, then build a fresh `memory_agent` and confirm it uses the new region. One value per key, latest wins.

In [26]:
# Two sessions, two memories. They do not leak into each other.
def remember_in(sid, text):
    a = Agent(model=model, tools=[current_time],
              session_manager=FileSessionManager(session_id=sid, storage_dir="session_store"),
              system_prompt=SYSTEM_PROMPT)
    return str(a(text))

remember_in("alice", "My name is Alice and my region is us-ord.")
remember_in("bob", "My name is Bob and my region is us-sea.")


print("ALICE:", remember_in("alice", "What is my name and region?"))
print("BOB:  ", remember_in("bob", "What is my name and region?"))

It's nice to meet you, Alice! Your region is `us-ord`, which stands for Chicago, Illinois data center. How can I assist you further?It's nice to meet you, Bob! Are you looking for specific information related to the us-sea region or Akamai services? Could you please provide more details so I can assist you better?Your name is Alice and your region is `us-ord` (Chicago, Illinois).ALICE: Your name is Alice and your region is `us-ord` (Chicago, Illinois).

Your name is Bob and your region is us-sea.BOB:   Your name is Bob and your region is us-sea.



## Summary

- A model is stateless. Memory exists only because something keeps and resends the past messages.
- In-process memory: reuse one agent. It remembers until the process exits.
- Durable file sessions: use a `FileSessionManager` to rehydrate a fresh agent from disk.
- Durable shared sessions: use an `S3SessionManager` on Akamai Object Storage so any replica behind a NodeBalancer can continue the same conversation.
- Long-term memory: remember the user, not just the conversation. Store preferences and facts with tools, then inject the most important memory into the system prompt so the agent has it every turn.

## Next

**Module 5: Diagrams that do not lie.** A small model will happily draw a cluster it never
read. Next you will give the agent deterministic tools that read real resources and render
them, so the picture is always true.